<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

<div class="alert alert-info alert-success">
<h3>Élaboration d'image composite MTG / FCI avec SATPY </h3></div>

## <a id='TOC-TOP'></a>Contenus

<div class="alert alert-block alert-warning">
    
<b>PREREQUIS </b>
    
Le kernel "env_MF_teledetection" doit être utilisé

### Ce Notebook fait suite au Notebook précédent : " H - Satpy : découverte et utilisation " , pour exercice

</div>
<hr>

<div class="alert alert-info" role="alert">

### <a id='section1'></a>1.Tout d'abord, il faut procéder à l'importation des librairies nécessaires.

</div>

In [ ]:
from satpy.scene import Scene
from satpy import find_files_and_readers
from datetime import datetime
import sys
from pyresample.geometry import AreaDefinition
import numpy as np
import os
from osgeo import gdal
from PIL import Image
import subprocess
from IPython.display import display,HTML
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

<div class="alert alert-info" role="alert">
    
### <a id='section2'></a>2.Définition du produit, date et heure, reader, et répertoires.

</div>

### Données d'entrée : fichiers chunk mtg (40) 
Pour télécharger d'autres fichiers, Cf. Jupyter Notebook  <a href="eumdac.ipynb" target="_blank"> Eumdac : téléchargement de données MTG</a></br>

Répertoires disponibles :
- /stockage/DATA/20260527/ -> 
- /stockage/DATA/202509271200/ -> Perturbation proche atlantique ouest Espagne

In [ ]:
input = '/stockage/DATA/20260527/'

In [ ]:
cd ~/MF_DATA_MANIPULATION

In [ ]:
download_dir = os.path.join(os.getcwd(), "RESULTS")
os.makedirs(download_dir, exist_ok=True)
output='RESULTS'

In [ ]:
!ls $input |tail

### Définition de la date et heure de la composite à créer

In [ ]:
annee='2026'
mois='05'
jour='27'
heure='16'
yyyy=int(annee)
mm=int(mois)
dd=int(jour)
hh_debut=int(heure)
min_debut=int('10')
hh_fin=int(heure)
min_fin=int('20')

<div class="alert alert-info" role="alert">
    
### <a id='section2'></a>3.Création du produit.

</div>

In [ ]:
composite_name = "cimss_cloud_type"   # Modifiez ici pour tester d’autres composites

In [ ]:
reader_to_use = "fci_l1c_nc"
filename = (output + '/RGB_satpy_' + composite_name )
myfiles = find_files_and_readers(base_dir=input,
                                 start_time=datetime(yyyy,mm,dd,hh_debut,min_debut),
                                 end_time=datetime(yyyy,mm,dd,hh_fin,min_fin),
                                 reader=reader_to_use)
scn = Scene(filenames=myfiles)
scn.load([composite_name], upper_right_corner='NE')
natscn = scn.resample(scn.coarsest_area(), resampler='nearest')
natscn.save_dataset(composite_name, filename=filename + '.tif')

!gdalwarp -overwrite -ts 1800 1800 {filename}.tif {filename}_redim.tif
taille_redim=700  # si nécessaire pour affichage plus léger, 1800x1800 par défaut
!convert  {filename}.tif {filename}_redim_vraie_res.jpg > /dev/null 2>&1
!convert  {filename}_redim.tif {filename}_redim_HD.jpg > /dev/null 2>&1
!convert -resize {taille_redim} {filename}_redim.tif {filename}_redim.jpg > /dev/null 2>&1
imgredim = Image.open(filename + '_redim.jpg')
display(imgredim)

Quelques liens utiles: </br>
<img src='../OUTILS/icon_pdf.png' alt='icon pdf' width='30px'></img><a href="../DOCS/Fiches_produits_MTG_legendes-RGB.pdf" target="_blank"> Principales légendes des produits RGB</a>

<div class="alert alert-info" role="alert">
    
### <a id='section2'></a>4.Création d'une image à partir d'une zone de découpe 

</div>

In [ ]:
### Définition de la zone, et redimmensionnement de l'image 

In [ ]:
#exemple Europe
coord_nord = 55
coord_sud = 40
coord_ouest = -10
coord_est = 15

In [ ]:
!gdalwarp -overwrite -t_srs "EPSG:4326" -te {coord_ouest} {coord_sud} {coord_est} {coord_nord}  {filename}.tif {filename}_decoupe_HD.tif
!convert -resize 1000 {filename}_decoupe_HD.tif {filename}_decoupe_min.jpg
im2 = Image.open(filename + '_decoupe_min.jpg')
display(im2)

### Ajout texte et logo ?

In [ ]:
!magick file_in -font OUTILS/dejavu-sans/ttf/DejaVuSans.ttf -pointsize 18 -fill '#ffffff'  -annotate +30+30 "texte_a_modifier_ici" flie_out

### Nettoyage

In [ ]:
!rm {output}/*xml #{output}/*.tif

In [ ]:
#!rm -f {output}/*


<div class="alert alert-info" role="alert">

### <a id='section1'></a>Exercice : créer une autre image composite RGB avec satpy et découper sur une autre zone avec GDAL. 
**<a href="./NOTEBOOK/MTG_satpy_chunk_exercice.ipynb" target="_blank"> Nouveau NoteBook exercice</a>**</br>

</div>